# 3. Analisi esplorative multivariate: PCA, MCA e FAMD

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedeGambe/Master_s_thesis_Data_science/blob/main/notebooks/03_analisi_esplorative.ipynb)

Versione Python degli script R (cartella `R/`). Le analisi sono svolte sui **soli possessori di BEV**, come nella tesi, per costruire gli indici sintetici usati poi nel clustering. I segni delle componenti possono essere invertiti rispetto a R: gli assi sono definiti a meno del segno.

In [ ]:
# Configurazione dell'ambiente (locale o Google Colab)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("Master_s_thesis_Data_science"):
        !git clone -q -b main https://github.com/FedeGambe/Master_s_thesis_Data_science.git
    %cd Master_s_thesis_Data_science
    !pip install -q -r requirements-colab.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import plotly.io as pio
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)
if os.environ.get("TESI_GRAFICI_STATICI"):
    pio.renderers.default = "png"  # usato solo per salvare output visibili su GitHub
    pio.defaults.default_width = 1000

In [ ]:
from tesi_bev import config as C, dati, esplorativa as E, grafici as G
bev = dati.solo_bev(dati.crea_variabili_categoriali())
print(f"Possessori di BEV: {len(bev)}")

## 3.1 Variabili categoriali derivate

Le variabili quantitative sono discretizzate con le soglie della tesi. Da notare che la classe *VMT molto alto*
contiene pochissime osservazioni, perché la soglia cade a metà tra il terzo quartile e il massimo.

In [ ]:
pd.concat({c: bev[c].value_counts() for c in ["VMT categoriale", "Sensibilità ambientale categoriale",
                                             "Numero viaggi lunghi categoriale"]}).to_frame("n")

## 3.2 PCA sulle variabili di mobilità

In [ ]:
pca = E.pca(bev, C.VARIABILI_MOBILITA)
print(f"Alpha di Cronbach: {E.alpha_cronbach(bev[C.VARIABILI_MOBILITA]):.3f}")
G.scree(pca["varianza"], "PCA mobilità: varianza spiegata").show()
pca["loadings"]

In [ ]:
G.loadings(pca['loadings'], 'PCA mobilità: loadings PC1-PC2').show()

L'alpha di Cronbach è bassa (≈0,33): le quattro variabili di mobilità misurano aspetti diversi e la prima componente
spiega solo un terzo della varianza. Gli indici *Viaggi PCA* e *Mobilità quotidiana PCA* vanno quindi interpretati
con cautela.

## 3.3 MCA tematiche

In [ ]:
mca = E.mca_gruppi(bev)
for nome, r in mca.items():
    print(f"MCA {nome}: Dim1+Dim2 = {r['autovalori']['cumulata_%'].iloc[1]:.1f}% dell'inerzia")
    G.mappa_categorie(r["categorie"], f"MCA {nome}").show()

In [ ]:
mca['sociodemografiche']['categorie'].sort_values('contributo_Dim1_%', ascending=False)

## 3.4 FAMD (variabili miste)

In [ ]:
famd = E.famd(bev, n_componenti=10)
G.scree(famd["autovalori"], "FAMD: varianza spiegata").show()
famd["contributi"][["contributo_Dim1_%", "contributo_Dim2_%"]].sort_values("contributo_Dim1_%", ascending=False)

## 3.5 Indici compositi

Le prime due dimensioni di ogni analisi diventano gli indici usati nel clustering. Correzione rispetto allo script R
originale: *Status socioeconomico* e *Distanza sostenibilità* erano copiati per errore dall'MCA sulla mobilità.

In [ ]:
E.indici_compositi(bev).describe()

## 3.6 Confronto con R

In [ ]:
from pathlib import Path
if Path("risultati/r/pca_mobilita_varianza.csv").exists():
    r_pca = pd.read_csv("risultati/r/pca_mobilita_varianza.csv", index_col=0)
    confronto = pd.DataFrame({"Python": pca["varianza"]["varianza_%"].values, "R": r_pca["varianza_perc"].values},
                             index=pca["varianza"].index)
    display(confronto)
else:
    print("Eseguire 'Rscript R/esegui_tutto.R' per il confronto con R")